# 01 · Data Validation
Live or synthetic price levels, data quality, correlation regimes.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

USE_LIVE = False

if USE_LIVE:
    try:
        from data.fetcher import load_em_universe
        from core.returns import log_returns
        raw = load_em_universe(start='2015-01-01')
        fx_prices = raw['fx']; eq_prices = raw['equity']
        fx_ret = log_returns(fx_prices); eq_ret = log_returns(eq_prices)
        DATA_SOURCE = 'Live'
    except Exception as exc:
        print(f'Live failed ({exc}) — synthetic'); USE_LIVE = False

if not USE_LIVE:
    from data.synthetic import generate_em_universe
    uni = generate_em_universe(seed=42)
    fx_prices = uni.prices_fx; eq_prices = uni.prices_eq
    fx_ret = uni.fx; eq_ret = uni.equity
    DATA_SOURCE = 'Synthetic'

print(f'Source : {DATA_SOURCE}')
print(f'FX     : {fx_prices.shape}  {fx_prices.index[0].date()} to {fx_prices.index[-1].date()}')


## 2 · Price Levels Table — last 10 rows with range flags


In [ ]:
FX_RANGES  = dict(CLP=(500,1200), BRL=(2.,8.), MXN=(10.,25.), COP=(1500,6000), PEN=(2.5,4.5))
EQ_RANGES  = dict(CHL=(1,500), BRA=(1,500), MEX=(1,500), COL=(1,500), PER=(1,500))

def range_table(df, ranges):
    rows = []
    for col in df.columns:
        s = df[col].dropna()
        if s.empty: continue
        lo, hi = ranges.get(col, (float('-inf'), float('inf')))
        latest = float(s.iloc[-1])
        rows.append({'Series': col, 'Latest Date': str(s.index[-1].date()),
                     'Latest Price': round(latest,4), 'Low': lo, 'High': hi,
                     'Pass': 'OK' if lo <= latest <= hi else 'FAIL'})
    return pd.DataFrame(rows).set_index('Series')

print('FX (local CCY per USD):'); print(range_table(fx_prices, FX_RANGES).to_string())
print(); print('Equity ETF (USD):');   print(range_table(eq_prices, EQ_RANGES).to_string())
print(); print('Last 10 FX rows:');    print(fx_prices.tail(10).round(4).to_string())


## 3 · Data Quality — NaN counts, gaps, annualized volatility


In [ ]:
def quality_report(returns_df, label):
    rows = []
    for col in returns_df.columns:
        s = returns_df[col]
        valid = s.dropna()
        if len(valid) < 2:
            rows.append({'Series': col, 'NaN%': round(s.isna().mean()*100,2), 'Max gap (d)': 0, 'Ann vol %': float('nan')})
            continue
        run_lens, cur = [], 0
        for v in s.isna():
            if v: cur += 1
            else:
                if cur: run_lens.append(cur)
                cur = 0
        if cur: run_lens.append(cur)
        rows.append({'Series': col, 'NaN%': round(s.isna().mean()*100,2),
                     'Max gap (d)': max(run_lens) if run_lens else 0,
                     'Ann vol %': round(float(valid.std()*np.sqrt(252)*100),1)})
    print(f'--- {label} ---'); print(pd.DataFrame(rows).set_index('Series').to_string())
    return pd.DataFrame(rows).set_index('Series')

dq_fx = quality_report(fx_ret, 'FX returns')
print(); dq_eq = quality_report(eq_ret, 'Equity returns')


## 4 · FX Price Levels (rebased to 100 at start)


In [ ]:
colors = ['#00d4aa','#e74c3c','#f39c12','#9b59b6','#3498db']
fig, ax = plt.subplots(figsize=(12, 4))
for i, col in enumerate(fx_prices.columns):
    s = fx_prices[col].dropna(); rebased = s / s.iloc[0] * 100
    ax.plot(rebased.index, rebased, label=col, color=colors[i], lw=1.3)
    ax.annotate(f'{col} {rebased.iloc[-1]:.1f}', xy=(rebased.index[-1], rebased.iloc[-1]),
                xytext=(5,0), textcoords='offset points', fontsize=8, color=colors[i], va='center')
ax.set_title('EM FX vs USD (rebased to 100)'); ax.set_ylabel('Index')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(loc='upper left', fontsize=9); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()


## 5 · Equity ETF Prices (rebased to 100)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for i, col in enumerate(eq_prices.columns):
    s = eq_prices[col].dropna(); rebased = s / s.iloc[0] * 100
    ax.plot(rebased.index, rebased, label=col, color=colors[i], lw=1.3)
    ax.annotate(f'{col} {rebased.iloc[-1]:.1f}', xy=(rebased.index[-1], rebased.iloc[-1]),
                xytext=(5,0), textcoords='offset points', fontsize=8, color=colors[i], va='center')
ax.set_title('EM Equity ETFs (rebased to 100)'); ax.set_ylabel('Index')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(loc='upper left', fontsize=9); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()


## 6 · Pairwise Correlation Heatmap — Calm vs Stress


In [ ]:
CALM_S, CALM_E    = '2018-01-01', '2019-12-31'
STRESS_S, STRESS_E = '2020-02-20', '2020-04-30'

panel_ret   = pd.concat([fx_ret, eq_ret], axis=1).dropna(how='all')
corr_calm   = panel_ret.loc[CALM_S:CALM_E].corr()
corr_stress = panel_ret.loc[STRESS_S:STRESS_E].corr()
corr_delta  = corr_stress - corr_calm

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
kw = dict(vmin=-1, vmax=1, cmap='RdBu_r', annot=True, fmt='.2f', linewidths=0.3, annot_kws={'size':7})
sns.heatmap(corr_calm,   ax=axes[0], **kw); axes[0].set_title(f'Calm ({CALM_S} to {CALM_E})')
sns.heatmap(corr_stress, ax=axes[1], **kw); axes[1].set_title('Stress (COVID)')
dkw = dict(vmin=-0.6, vmax=0.6, cmap='RdYlGn_r', annot=True, fmt='+.2f', linewidths=0.3, annot_kws={'size':7})
sns.heatmap(corr_delta,  ax=axes[2], **dkw); axes[2].set_title('Delta (Stress minus Calm)')
plt.tight_layout(); plt.show()
mask = ~np.eye(len(corr_calm), dtype=bool)
print(f'Avg off-diag corr  Calm:   {corr_calm.where(mask).stack().mean():.3f}')
print(f'Avg off-diag corr  Stress: {corr_stress.where(mask).stack().mean():.3f}')


## 7 · Manual Spot-Check — price vs log-return consistency


In [ ]:
loc = fx_prices.index.get_indexer([pd.Timestamp('2020-03-20')], method='nearest')[0]
audit_date = fx_prices.index[loc]
CHECK_SERIES = fx_prices.columns[0]
stored = float(fx_prices.iloc[loc][CHECK_SERIES])
print(f'Check date: {audit_date.date()}   Series: {CHECK_SERIES}')
print(f'Stored price: {stored:.4f}')
if loc > 0 and CHECK_SERIES in fx_ret.columns and audit_date in fx_ret.index:
    prev  = float(fx_prices.iloc[loc-1][CHECK_SERIES])
    lr    = float(fx_ret.loc[audit_date, CHECK_SERIES])
    recon = prev * np.exp(lr)
    diff  = abs(stored - recon)
    print(f'Reconstructed: {recon:.4f}   Diff: {diff:.6f}')
    print('Consistency: OK' if diff < 0.01 else 'Consistency: DISCREPANCY')


## 8 · Export to Excel


In [ ]:
from datetime import date
os.makedirs('../exports', exist_ok=True)
filename = f'../exports/01_data_validation_{date.today()}.xlsx'
with __import__('openpyxl') and pd.ExcelWriter(filename, engine='openpyxl') as writer:
    fx_prices.tail(252).to_excel(writer, sheet_name='FX Prices')
    eq_prices.tail(252).to_excel(writer, sheet_name='Equity Prices')
    dq_fx.to_excel(writer, sheet_name='Data Quality FX')
    dq_eq.to_excel(writer, sheet_name='Data Quality EQ')
    fx_ret.tail(252).to_excel(writer, sheet_name='FX Returns')
    eq_ret.tail(252).to_excel(writer, sheet_name='Equity Returns')
    corr_calm.to_excel(writer, sheet_name='Correlation Calm')
    corr_stress.to_excel(writer, sheet_name='Correlation Stress')
print(f'Exported: {filename}')
